# PyBullet Ping-Pong Ball Bounce: Multi-View Video Capture

This notebook uses the local `phys_sim` virtual environment, PyBullet in SI units, and TinyRenderer camera captures to simulate a ping-pong ball bouncing on a tabletop. It includes gravity, rigid-body contact, restitution, friction, and quadratic air drag, then records synchronized MP4 videos from front, side, back, and top views.

In [ ]:
import os
import sys
from pathlib import Path

# This notebook is intended to run with the repository-local virtual environment.
assert "phys_sim" in sys.executable, (
    f"Expected the phys_sim virtual environment, but got: {sys.executable}\n"
    "In Jupyter, select the kernel named 'phys_sim'."
)

import numpy as np
import pybullet as p
import pybullet_data
import imageio.v2 as imageio
from IPython.display import Video, display

print(f"Python executable: {sys.executable}")
print(f"PyBullet data path: {pybullet_data.getDataPath()}")

## Scenario Parameters

All distances are meters, mass is kilograms, and time is seconds. The ball uses regulation ping-pong dimensions: 40 mm diameter and 2.7 g mass. The default bounce starts from rest vertically with a tiny lateral drift so the top camera is useful.

In [ ]:
OUTPUT_DIR = Path("pybullet_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Simulation fidelity
GRAVITY = -9.80665
SIM_HZ = 480
TIME_STEP = 1.0 / SIM_HZ
VIDEO_FPS = 60
STEPS_PER_FRAME = SIM_HZ // VIDEO_FPS
DURATION_SEC = 4.0
N_STEPS = int(DURATION_SEC * SIM_HZ)

# Table geometry: tabletop center z plus half thickness gives the upper surface.
TABLE_LENGTH = 1.20
TABLE_WIDTH = 0.80
TABLE_THICKNESS = 0.06
TABLE_TOP_Z = 0.75
TABLE_CENTER_Z = TABLE_TOP_Z - TABLE_THICKNESS / 2.0

# Regulation ping-pong ball geometry and material.
BALL_RADIUS = 0.020
BALL_MASS = 0.0027
BALL_START_Z = TABLE_TOP_Z + 0.80
BALL_START_XY = [-0.12, -0.04]
BALL_INITIAL_LINEAR_VELOCITY = [0.08, 0.03, 0.0]
BALL_INITIAL_ANGULAR_VELOCITY = [0.0, 0.0, 0.0]

# A regulation ball dropped from 305 mm should rebound about 240-260 mm on a standard block,
# which implies an effective coefficient of restitution near sqrt(0.25 / 0.305) ~= 0.91.
BALL_RESTITUTION = 0.93
BALL_LATERAL_FRICTION = 0.20
BALL_ROLLING_FRICTION = 0.0005
BALL_SPINNING_FRICTION = 0.0005

TABLE_RESTITUTION = 0.93
TABLE_LATERAL_FRICTION = 0.35

# Aerodynamics matter for a very light 40 mm ball.
AIR_DENSITY = 1.225
DRAG_COEFFICIENT = 0.47
BALL_CROSS_SECTION = np.pi * BALL_RADIUS ** 2

# Rendering
IMG_WIDTH = 960
IMG_HEIGHT = 544

print(f"Writing videos and trajectory files to: {OUTPUT_DIR.resolve()}")

In [ ]:
def connect_pybullet():
    """Create a clean DIRECT PyBullet connection for headless notebook rendering."""
    if p.isConnected():
        p.disconnect()
    client = p.connect(p.DIRECT)
    p.resetSimulation(physicsClientId=client)
    p.setAdditionalSearchPath(pybullet_data.getDataPath(), physicsClientId=client)
    p.setGravity(0, 0, GRAVITY, physicsClientId=client)
    p.setTimeStep(TIME_STEP, physicsClientId=client)
    p.setPhysicsEngineParameter(
        fixedTimeStep=TIME_STEP,
        numSolverIterations=150,
        numSubSteps=2,
        deterministicOverlappingPairs=1,
        physicsClientId=client,
    )
    return client


def create_tabletop_world(client):
    """Build a rigid tabletop, floor, and spherical ball in SI units."""
    p.loadURDF("plane.urdf", physicsClientId=client)

    table_collision = p.createCollisionShape(
        p.GEOM_BOX,
        halfExtents=[TABLE_LENGTH / 2, TABLE_WIDTH / 2, TABLE_THICKNESS / 2],
        physicsClientId=client,
    )
    table_visual = p.createVisualShape(
        p.GEOM_BOX,
        halfExtents=[TABLE_LENGTH / 2, TABLE_WIDTH / 2, TABLE_THICKNESS / 2],
        rgbaColor=[0.58, 0.40, 0.24, 1.0],
        physicsClientId=client,
    )
    table_id = p.createMultiBody(
        baseMass=0,
        baseCollisionShapeIndex=table_collision,
        baseVisualShapeIndex=table_visual,
        basePosition=[0, 0, TABLE_CENTER_Z],
        physicsClientId=client,
    )
    p.changeDynamics(
        table_id,
        -1,
        restitution=TABLE_RESTITUTION,
        lateralFriction=TABLE_LATERAL_FRICTION,
        physicsClientId=client,
    )

    ball_collision = p.createCollisionShape(
        p.GEOM_SPHERE,
        radius=BALL_RADIUS,
        physicsClientId=client,
    )
    ball_visual = p.createVisualShape(
        p.GEOM_SPHERE,
        radius=BALL_RADIUS,
        rgbaColor=[0.08, 0.38, 0.90, 1.0],
        physicsClientId=client,
    )
    ball_id = p.createMultiBody(
        baseMass=BALL_MASS,
        baseCollisionShapeIndex=ball_collision,
        baseVisualShapeIndex=ball_visual,
        basePosition=[BALL_START_XY[0], BALL_START_XY[1], BALL_START_Z],
        physicsClientId=client,
    )
    p.changeDynamics(
        ball_id,
        -1,
        restitution=BALL_RESTITUTION,
        lateralFriction=BALL_LATERAL_FRICTION,
        rollingFriction=BALL_ROLLING_FRICTION,
        spinningFriction=BALL_SPINNING_FRICTION,
        physicsClientId=client,
    )
    p.resetBaseVelocity(
        ball_id,
        linearVelocity=BALL_INITIAL_LINEAR_VELOCITY,
        angularVelocity=BALL_INITIAL_ANGULAR_VELOCITY,
        physicsClientId=client,
    )

    # Give the renderer a directional light that makes the table/ball readable.
    p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0, physicsClientId=client)
    return table_id, ball_id


def apply_ping_pong_air_drag(client, ball_id):
    """Apply quadratic drag: Fd = -0.5 * rho * Cd * A * |v| * v."""
    lin_vel, _ = p.getBaseVelocity(ball_id, physicsClientId=client)
    velocity = np.asarray(lin_vel, dtype=float)
    speed = np.linalg.norm(velocity)
    if speed == 0:
        return
    drag = -0.5 * AIR_DENSITY * DRAG_COEFFICIENT * BALL_CROSS_SECTION * speed * velocity
    pos, _ = p.getBasePositionAndOrientation(ball_id, physicsClientId=client)
    p.applyExternalForce(
        ball_id,
        -1,
        drag.tolist(),
        pos,
        flags=p.WORLD_FRAME,
        physicsClientId=client,
    )


def camera_matrices(view_name):
    target = [0, 0, TABLE_TOP_Z + 0.20]
    cameras = {
        "front": {"eye": [0.0, -1.65, 1.15], "up": [0, 0, 1], "fov": 48},
        "side": {"eye": [1.65, 0.0, 1.15], "up": [0, 0, 1], "fov": 48},
        "back": {"eye": [0.0, 1.65, 1.15], "up": [0, 0, 1], "fov": 48},
        "top": {"eye": [0.0, 0.0, 2.65], "up": [0, 1, 0], "fov": 42},
    }
    spec = cameras[view_name]
    view = p.computeViewMatrix(spec["eye"], target, spec["up"])
    proj = p.computeProjectionMatrixFOV(
        fov=spec["fov"],
        aspect=IMG_WIDTH / IMG_HEIGHT,
        nearVal=0.02,
        farVal=5.0,
    )
    return view, proj


CAMERA_NAMES = ["front", "side", "back", "top"]
CAMERA_MATRICES = {name: camera_matrices(name) for name in CAMERA_NAMES}


def render_rgb(client, view_name):
    view, proj = CAMERA_MATRICES[view_name]
    _, _, rgba, _, _ = p.getCameraImage(
        width=IMG_WIDTH,
        height=IMG_HEIGHT,
        viewMatrix=view,
        projectionMatrix=proj,
        renderer=p.ER_TINY_RENDERER,
        lightDirection=[-0.4, -0.6, -1.0],
        physicsClientId=client,
    )
    rgba = np.asarray(rgba, dtype=np.uint8).reshape(IMG_HEIGHT, IMG_WIDTH, 4)
    return rgba[:, :, :3]

## Run Simulation and Record Four Videos

This cell advances the physics simulation at 240 Hz and records at 60 FPS. The four cameras are sampled at the same simulation ticks, so the videos are time-aligned.

In [ ]:
client = connect_pybullet()
table_id, ball_id = create_tabletop_world(client)

video_paths = {name: OUTPUT_DIR / f"ping_pong_bounce_{name}.mp4" for name in CAMERA_NAMES}
writers = {
    name: imageio.get_writer(
        path,
        fps=VIDEO_FPS,
        codec="libx264",
        quality=8,
        macro_block_size=16,
    )
    for name, path in video_paths.items()
}

trajectory = []

try:
    for step in range(N_STEPS + 1):
        t = step * TIME_STEP
        pos, orn = p.getBasePositionAndOrientation(ball_id, physicsClientId=client)
        lin_vel, ang_vel = p.getBaseVelocity(ball_id, physicsClientId=client)
        contacts = p.getContactPoints(bodyA=ball_id, bodyB=table_id, physicsClientId=client)
        trajectory.append([
            t,
            pos[0], pos[1], pos[2],
            lin_vel[0], lin_vel[1], lin_vel[2],
            len(contacts),
        ])

        if step % STEPS_PER_FRAME == 0:
            for name, writer in writers.items():
                writer.append_data(render_rgb(client, name))

        apply_ping_pong_air_drag(client, ball_id)
        p.stepSimulation(physicsClientId=client)
finally:
    for writer in writers.values():
        writer.close()
    p.disconnect(client)

trajectory = np.asarray(trajectory)
trajectory_path = OUTPUT_DIR / "ping_pong_bounce_trajectory.csv"
np.savetxt(
    trajectory_path,
    trajectory,
    delimiter=",",
    header="time,x,y,z,vx,vy,vz,num_table_contacts",
    comments="",
)

print("Saved videos:")
for name, path in video_paths.items():
    print(f"  {name:>5}: {path}")
print(f"Saved trajectory: {trajectory_path}")

## Quick Bounce Sanity Check

This reports impact times, rebound peaks, and approximate peak-height ratios. Because this is a ping-pong ball, the simulation includes air drag, so the vacuum contact time is only a reference.

In [ ]:
times = trajectory[:, 0]
z = trajectory[:, 3]
vacuum_contact_time = np.sqrt(2 * ((BALL_START_Z - BALL_RADIUS) - TABLE_TOP_Z) / abs(GRAVITY))

def local_maxima(values):
    return np.flatnonzero((values[1:-1] > values[:-2]) & (values[1:-1] >= values[2:])) + 1

def local_minima(values):
    return np.flatnonzero((values[1:-1] < values[:-2]) & (values[1:-1] <= values[2:])) + 1

surface_center_z = TABLE_TOP_Z + BALL_RADIUS
impact_indices = local_minima(z)
impact_indices = impact_indices[z[impact_indices] <= surface_center_z + 0.01]
peak_indices = local_maxima(z)
rebound_peak_indices = peak_indices[times[peak_indices] > times[impact_indices[0]]] if len(impact_indices) else []
peak_heights = z[rebound_peak_indices] - (TABLE_TOP_Z + BALL_RADIUS)
peak_times = times[rebound_peak_indices]

if len(impact_indices):
    first_impact_time = times[impact_indices[0]]
    min_z = np.min(z)

    print(f"Vacuum first-contact estimate:    {vacuum_contact_time:.4f} s")
    print(f"First simulated impact:            {first_impact_time:.4f} s")
    print(f"Lowest ball center height:         {min_z:.4f} m")
    print(f"Tabletop + ball radius:            {TABLE_TOP_Z + BALL_RADIUS:.4f} m")
    print(f"Detected impacts:                  {len(impact_indices)}")

    print("\nImpact times:")
    for i, idx in enumerate(impact_indices[:8], start=1):
        print(f"  impact {i}: t={times[idx]:.3f} s, center_z={z[idx]:.4f} m")

    print("\nRebound peaks above the tabletop:")
    for i, (t_peak, h_peak) in enumerate(zip(peak_times[:6], peak_heights[:6]), start=1):
        print(f"  peak {i}: t={t_peak:.3f} s, height={h_peak:.3f} m")

    if len(peak_heights) >= 2:
        ratios = peak_heights[1:6] / peak_heights[:5]
        print("\nSuccessive peak-height ratios:", np.round(ratios, 3))
else:
    print("No table contact detected. Increase DURATION_SEC or lower BALL_START_Z.")

## Preview Videos

In [ ]:
for name in CAMERA_NAMES:
    print(name)
    display(Video(str(video_paths[name]), embed=True, html_attributes="controls loop"))

## Notes for Tuning

- Increase `SIM_HZ`, `numSolverIterations`, or `numSubSteps` for more stable contact resolution.
- Change `BALL_RESTITUTION`, `TABLE_RESTITUTION`, and friction constants to match a specific ball/table material pair.
- Change `DRAG_COEFFICIENT` or `AIR_DENSITY` if you want a different aerodynamic model.
- Increase `IMG_WIDTH`, `IMG_HEIGHT`, or `quality` for cleaner videos, at the cost of slower rendering.
- The generated CSV contains time, position, velocity, and contact counts for downstream analysis.